## Data Loading

In [1]:
from datasets import load_dataset

dataset_raw = load_dataset("lfcc/portuguese_ner")
dataset_raw

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 930/930 [00:00<00:00, 384608.83 examples/s]


DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3716
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 930
    })
})

In [2]:
dataset_raw["train"].features

{'tokens': List(Value('string')),
 'ner_tags': List(ClassLabel(names=['O', 'B-Data', 'I-Data', 'B-Local', 'I-Local', 'B-Organizacao', 'I-Organizacao', 'B-Pessoa', 'I-Pessoa', 'B-Profissao', 'I-Profissao']))}

## Data Pre-Processing

In [3]:
from transformers import AutoTokenizer

#usar o tokenizer do modelo que quero utilizar
tokenizer = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")

In [4]:
inputs = tokenizer("As aulas de PLNEB são muito interessantes!")
inputs

{'input_ids': [101, 510, 6880, 125, 212, 22327, 22320, 19591, 453, 785, 20764, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [5]:
#converter os ids para tokens - "partir" as palavras em subwords que o modelo conhece
tokens=tokenizer.convert_ids_to_tokens(inputs["input_ids"])
print(tokens)
# cls token é utilizado para o treino do modelo , sep token determina o fim de uma frase (herança do treino do modelo)

['[CLS]', 'As', 'aulas', 'de', 'P', '##L', '##N', '##EB', 'são', 'muito', 'interessantes', '!', '[SEP]']


In [6]:
dataset_raw["train"]["tokens"]

Column([['Filiação', ':', 'Antonio', 'Joaquim', 'Aguiar', 'e', 'Engracia', 'Maria', '.', 'Natural', 'e/ou', 'residente', 'em', 'CUNHA', ',', 'Santa', 'Maria', ',', 'actual', 'concelho', 'de', 'PAREDES', 'COURA', 'e', 'distrito', '(', 'ou', 'país', ')', 'Viana', 'do', 'Castelo', '.'], ['Filiação', ':', 'Domingos', 'Pires', 'e', 'Comba', 'Fernandes', '.', 'Natural', 'e/ou', 'residente', 'em', 'VALONGO', 'MILHAIS', ',', 'Sao', 'Goncalo', ',', 'actual', 'concelho', 'de', 'MURCA', 'e', 'distrito', '(', 'ou', 'país', ')', 'VILA', 'REAL', '.'], ['Termo', 'de', 'justificação', 'do', 'baptismo', 'de', 'Pedro', 'Gonçalves', 'Coques', ',', 'nascido', 'em', '29.06.1876', 'e', 'baptizado', '"', '(', '…', ')', 'por', 'dias', 'do', 'mês', 'de', 'Julho', 'do', 'dito', 'ano', ',', '(', '…', ')', '"', ',', 'na', 'igreja', 'do', 'Jardim', 'do', 'Mar', ',', 'Calheta', '.'], ['Doc.danificado', '.'], ['1898-11-01', '/', '1898-11-01'], ...])

In [7]:
tokens = ["as", "aulas","plneb", "são","interessantes","!"]
inputs = tokenizer(tokens, is_split_into_words=True) #percebe que ja é uma lista de tokens

new_tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"])
print(new_tokens)

['[CLS]', 'as', 'aulas', 'pl', '##ne', '##b', 'são', 'interessantes', '!', '[SEP]']


In [8]:
len(tokens), len(new_tokens)

(6, 10)

In [9]:
# correspondencia direta entre os novos tokens e os tokens originais, faz correspondecia pelos indices dos tokens na lista de tokens originais - NONE quando nao corresponde a nenhum
inputs.word_ids()

[None, 0, 1, 2, 2, 2, 3, 4, 5, None]

In [10]:
def align_labels_with_tokens(word_ids, labels):
    new_labels = []
    previous_word = None
    # para cada um destes elementos - [None, 0, 1, 2, 2, 2, 3, 4, 5, None]
    for word_id in word_ids:
        if word_id == None:
            new_labels.append(-100) # -100 é o codigo que o bert usa para ignorar a label
        elif previous_word != word_id:
            new_labels.append(labels[word_id]) #mantem a label
        else:
            new_labels.append(-100)
        previous_word = word_id
    return new_labels

def tokenize_dataset(dataset):
    res = []
    for row in dataset:
        inputs = tokenizer(row["tokens"], is_split_into_words=True, truncation=True,
            max_length=512)
        new_labels = align_labels_with_tokens(inputs.word_ids(), row["ner_tags"])
        inputs["labels"] = new_labels
        res.append(inputs)
    return res



train_data = tokenize_dataset(dataset_raw["train"])
test_data = tokenize_dataset(dataset_raw["test"])

len(train_data),len(test_data)


(3716, 930)

In [11]:
from datasets import Dataset
train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 3716
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 930
})


## Model Training

In [12]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained("neuralmind/bert-base-portuguese-cased")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 43922.92it/s]
[transformers] BertForTokenClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes

In [13]:
label_list = dataset_raw["train"].features["ner_tags"].feature.names
label_list

['O',
 'B-Data',
 'I-Data',
 'B-Local',
 'I-Local',
 'B-Organizacao',
 'I-Organizacao',
 'B-Pessoa',
 'I-Pessoa',
 'B-Profissao',
 'I-Profissao']

In [14]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [15]:
!pip install seqeval

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16252 sha256=19f491d304d5c415eb86487cb85c91e703c5239e1ed79a82051a74cc62607e38
  Stored in directory: /Users/guilhermefernandes/Library/Caches/pip/wheels/1a/67/4a/ad4082dd7dfc30f2abfe4d80a2ed5926a506eb8a972b4767fa
Successfully built seqeval

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


### EVALUATE

In [16]:
import evaluate

seqeval = evaluate.load("seqeval")

In [17]:
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

### TRAIN

In [18]:
id2label = {
    0: "O",
    1: "B-Data",
    2: "I-Data",
    3: "B-Local",
    4: "I-Local",
    5: "B-Organizacao",
    6: "I-Organizacao",
    7: "B-Pessoa",
    8: "I-Pessoa",
    9: "B-Profissao",
    10: "I-Profissao"
}

label2id = {
    "O": 0,
    "B-Data": 1,
    "I-Data": 2,
    "B-Local": 3,
    "I-Local": 4,
    "B-Organizacao": 5,
    "I-Organizacao": 6,
    "B-Pessoa": 7,
    "I-Pessoa": 8,
    "B-Profissao": 9,
    "I-Profissao": 10
}

In [19]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained(
    "neuralmind/bert-base-portuguese-cased",
    num_labels=11,
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 42208.72it/s]
[transformers] BertForTokenClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes

In [20]:
!pip install transformers[torch]

zsh:1: no matches found: transformers[torch]


In [21]:
training_args = TrainingArguments(
    output_dir="TPC_modelo_ner",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


RuntimeError: MPS backend out of memory (MPS allocated: 16.16 GiB, other allocations: 3.90 GiB, max allowed: 20.13 GiB). Tried to allocate 87.29 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

### Teste

In [ ]:
texto = """ A engenheira Maria Teresa, diretora da TechPortugal, anunciou ontem em Lisboa que a nova sede da empresa será inaugurada no dia 15 de agosto de 2026. O evento contará com a presença do Ministro da Economia no Centro de Congressos. """

In [ ]:
from transformers import pipeline

classifier = pipeline("ner",model=model, tokenizer=tokenizer, aggregation_strategy="first")
classifier(texto)

[{'entity_group': 'Profissao',
  'score': np.float32(0.6747537),
  'word': 'engenheira',
  'start': 3,
  'end': 13},
 {'entity_group': 'Pessoa',
  'score': np.float32(0.95026433),
  'word': 'Maria Teresa',
  'start': 14,
  'end': 26},
 {'entity_group': 'Organizacao',
  'score': np.float32(0.5298544),
  'word': 'TechPortugal',
  'start': 40,
  'end': 52},
 {'entity_group': 'Local',
  'score': np.float32(0.9260142),
  'word': 'Lisboa',
  'start': 72,
  'end': 78},
 {'entity_group': 'Data',
  'score': np.float32(0.8778305),
  'word': '15 de agosto de 2026',
  'start': 129,
  'end': 149},
 {'entity_group': 'Profissao',
  'score': np.float32(0.77054304),
  'word': 'Ministro da Economia',
  'start': 186,
  'end': 206}]